# 06 Evaluation following Ichmoukhamedov et al. (2024)

Implements the automated evaluation framework from:
Ichmoukhamedov, Hinns and Martens (2024). How good is my story? Towards quantitative
metrics for evaluating LLM generated XAI narratives. arXiv:2412.10220

The framework has three evaluation categories (Fig. 1 in the paper):

| Category | Metric | Status |
|----------|--------|--------|
| Faithfulness | Rank Accuracy (RA), Sign Accuracy (SA), Value Accuracy (VA) | implemented, selection bias |
| Assumptions | extraction by LLM + perplexity | extraction yes, PPL needs a local LLM |
| Human Similarity | cosine similarity to reference narratives | no reference narratives available |

Flow (Fig. 1):
1. Load narratives from pipelines 04/05/06
2. The extraction LLM (Claude) extracts per feature: rank, sign, value, assumption
3. Compare with the SHAP ground truth (Eq. 1 in the paper) to RA, SA, VA
4. Show assumptions (perplexity: optional call with a local LLM)

Adaptations to our scenario:
* Regression task (Poisson) instead of binary classification
* Contributions in log space, so sign: +1 raises, -1 lowers the prediction
* Feature values partly normalised (temp/hum/windspeed): VA with a denormalisation tolerance

In [1]:
from __future__ import annotations

import sys, json, re, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display

from utils import INSTANCE_IDS, RESULTS_DIR, EXPLANATIONS_DIR
from utils.explanations import FEATURE_SCHEMA
from utils.llm import ask_text, DEFAULT_MODEL

LOSS_KEY        = 'poisson_log'
MODEL           = DEFAULT_MODEL   # extraction LLM (paper: gpt-4o)
TOP_K           = 4               # paper: top 4 features by absolute contribution
PIPELINES       = ['04', '05', '06']
PIPELINE_LABELS = {'04': 'JSON to Text', '05': 'Vision', '06': 'Tool Use'}
XAI_MODELS     = ['xgb', 'ebm']

OUT_DIR = RESULTS_DIR / 'eval06_ichmoukhamedov'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Extraction model:  {MODEL}')
print(f'Top K features:    {TOP_K}  (paper: 4)')
print(f'Output:            {OUT_DIR}')

Extraction model:  claude-sonnet-4-6
Top K features:    4  (paper: 4)
Output:            /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/eval06_ichmoukhamedov


## 1. Load data

In [2]:
records = []
for pipeline in PIPELINES:
    p = RESULTS_DIR / f'pipeline{pipeline}'
    for xai in XAI_MODELS:
        for iid in INSTANCE_IDS:
            f = p / f'{xai}_inst{iid}.json'
            if not f.exists():
                print(f'MISSING: {f}'); continue
            d = json.loads(f.read_text())

            gt_path = EXPLANATIONS_DIR / f'local_{xai}_{LOSS_KEY}_inst{iid}.json'
            gt = json.loads(gt_path.read_text())

            records.append({
                'pipeline':          pipeline,
                'pipeline_label':    PIPELINE_LABELS[pipeline],
                'xai_model':         xai.upper(),
                'instance_id':       iid,
                'explanation':       d.get('explanation', ''),
                'gt_contributions':  gt['contributions'][:TOP_K],
                'gt_feature_values': gt['feature_values'],
                'gt_prediction':     gt['prediction'],
                'gt_y_true':         gt['y_true'],
            })

df = pd.DataFrame(records)
print(f'{len(df)} narratives loaded '
      f'({len(PIPELINES)} pipelines x {len(XAI_MODELS)} models x {len(INSTANCE_IDS)} instances)')
print(f'Top {TOP_K} ground truth features (example, first record):')
for c in records[0]['gt_contributions']:
    print(f"  rank={records[0]['gt_contributions'].index(c)}  "
          f"{c['feature']:12s}  contrib={c['contribution']:+.4f}  val={c['value']}")

60 narratives loaded (3 pipelines x 2 models x 10 instances)
Top 4 ground truth features (example, first record):
  rank=0  temp          contrib=-0.5488  val=0.2
  rank=1  hr            contrib=+0.4139  val=19.0
  rank=2  mnth          contrib=-0.3183  val=2.0
  rank=3  yr            contrib=-0.2938  val=0.0


## 2. Extraction LLM (Sec. III in the paper)

The paper proposes a separate extraction LLM that extracts per narrative (Fig. 3):
* Rank (rank): 0 based importance rank according to the narrative
* Sign (sign): +1 (raising) or -1 (lowering)
* Value (value): explicitly named feature value (or null)
* Assumption (assumption): injected background knowledge (one sentence or "None")

In [3]:
EXTRACTION_SYSTEM = (
    "You are an extraction model for XAI narratives of a bike rental model.\n"
    "Extract the requested information only from the narrative.\n"
    "Answer only with a valid JSON object, no text, no markdown blocks."
)

# Denormalisation: normalised feature values to human readable units
# (bike sharing: temp/41=C, hum/100=%, windspeed/67=km/h)
_DENORM = {
    'temp':      lambda v: v * 41,
    'hum':       lambda v: v * 100,
    'windspeed': lambda v: v * 67,
}


def build_extraction_prompt(explanation: str, xai_model: str) -> str:
    feat_descs = {f: FEATURE_SCHEMA[f]['description'] for f in FEATURE_SCHEMA}
    payload = {
        'task': (
            f'Extract structured information from the following English narrative '
            f'about a {xai_model} regression model for a bike rental company. '
            f'The model predicts hourly bike rentals. '
            f'Positive contributions raise the prediction, negative ones lower it.'
        ),
        'narrative': explanation,
        'all_features': list(FEATURE_SCHEMA.keys()),
        'feature_descriptions': feat_descs,
        'extraction_instruction': (
            'For each feature mentioned as important in the narrative, return an object:\n'
            '  rank: 0 based importance rank according to the narrative (0 = most important feature)\n'
            '  sign: +1 if the feature raises the prediction, -1 if it lowers it\n'
            '  value: numeric feature value if explicitly named in the narrative, else null\n'
            '  assumption: a single sentence of background knowledge why the feature has this effect; '
            '"None" if no background knowledge was added'
        ),
        'output_format_example': {
            'hr':   {'rank': 0, 'sign':  1, 'value': 13,   'assumption': 'Midday is typically high demand.'},
            'temp': {'rank': 1, 'sign': -1, 'value': None,  'assumption': 'Cold deters cyclists.'},
        },
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


def parse_extraction(raw: str) -> dict:
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if not m:
        return {}
    try:
        return json.loads(m.group())
    except json.JSONDecodeError:
        return {}


print('Extraction prompt and parser defined.')
sample = build_extraction_prompt(df.iloc[0]['explanation'][:200] + '...', 'XGB')
print(f'Prompt size (example): {len(sample)} characters')

Extraction prompt and parser defined.
Prompt size (example): 1830 characters


## 3. Run the LLM extractions

Already computed extractions are loaded from eval06_ichmoukhamedov/extractions.json (cache).

In [4]:
CACHE_PATH = OUT_DIR / 'extractions.json'

if CACHE_PATH.exists():
    extractions = json.loads(CACHE_PATH.read_text())
    print(f'Extractions loaded from cache: {len(extractions)} entries.')
else:
    extractions = {}

missing = [
    row for _, row in df.iterrows()
    if f"{row['pipeline']}_{row['xai_model'].lower()}_inst{row['instance_id']}" not in extractions
]

if missing:
    print(f'{len(missing)} missing extractions, starting API calls ...')
    total_in, total_out = 0, 0

    for row in missing:
        key = f"{row['pipeline']}_{row['xai_model'].lower()}_inst{row['instance_id']}"
        prompt = build_extraction_prompt(row['explanation'], row['xai_model'])

        response = ask_text(
            prompt,
            system=EXTRACTION_SYSTEM,
            model=MODEL,
            max_tokens=700,
            cache_system=True,
        )
        usage   = response.get('usage', {})
        in_tok  = usage.get('input_tokens', 0)
        out_tok = usage.get('output_tokens', 0)
        total_in += in_tok; total_out += out_tok

        raw = response['content'][0]['text'].strip()
        extraction = parse_extraction(raw)

        extractions[key] = {
            'pipeline_label': row['pipeline_label'],
            'xai_model':      row['xai_model'],
            'instance_id':    row['instance_id'],
            'raw':            raw,
            'extraction':     extraction,
            'usage':          {'input_tokens': in_tok, 'output_tokens': out_tok},
        }
        print(f"  {key}: {len(extraction)} features  in={in_tok}  out={out_tok}")

    CACHE_PATH.write_text(json.dumps(extractions, indent=2, ensure_ascii=False))
    print(f'\nSaved: {CACHE_PATH}')
    print(f'Total: input={total_in}  output={total_out}')
else:
    print('All extractions already cached.')

print(f'\nExample extraction (pipeline 04, XGB):')
sample_key = f'04_xgb_inst{INSTANCE_IDS[0]}'
if sample_key in extractions:
    print(json.dumps(extractions[sample_key]['extraction'], indent=2, ensure_ascii=False))

60 missing extractions, starting API calls ...
  04_xgb_inst224: 6 features  in=925  out=358
  04_xgb_inst580: 6 features  in=938  out=342
  04_xgb_inst1041: 6 features  in=926  out=349
  04_xgb_inst1481: 5 features  in=969  out=295
  04_xgb_inst1677: 6 features  in=940  out=348
  04_xgb_inst2058: 6 features  in=993  out=340
  04_xgb_inst2510: 5 features  in=960  out=300
  04_xgb_inst3543: 6 features  in=936  out=369
  04_xgb_inst3847: 6 features  in=949  out=348
  04_xgb_inst4454: 6 features  in=990  out=369
  04_ebm_inst224: 6 features  in=955  out=349
  04_ebm_inst580: 5 features  in=920  out=281
  04_ebm_inst1041: 6 features  in=962  out=343
  04_ebm_inst1481: 5 features  in=940  out=298
  04_ebm_inst1677: 6 features  in=944  out=372
  04_ebm_inst2058: 6 features  in=963  out=350
  04_ebm_inst2510: 5 features  in=958  out=287
  04_ebm_inst3543: 6 features  in=948  out=355
  04_ebm_inst3847: 6 features  in=943  out=343
  04_ebm_inst4454: 6 features  in=922  out=360
  05_xgb_inst224:

## 4. Faithfulness metrics (Eq. 1 in the paper)

$$X_A = \frac{\sum_{x_j \neq \phi} \delta_{x_j, x_j^*}}{n - \sum 1[x_j = \phi]}$$

* RA (Rank Accuracy): does the extracted rank match the actual SHAP rank?
* SA (Sign Accuracy): does the extracted sign match the SHAP sign?
* VA (Value Accuracy): does the extracted feature value match the actual value?
  Values not named in the narrative (null) are not scored.

Adaptation: feature values are compared both normalised (0 to 1) and denormalised
(C, %), because LLMs often quote the values in human readable form.

In [5]:
def is_value_match(feat: str, extracted: float, gt: float, tol: float = 1.0) -> bool:
    """Compare with tolerance; also checks denormalised units (for example C instead of [0,1])."""
    if abs(extracted - gt) <= tol:
        return True
    if feat in _DENORM:
        dv = _DENORM[feat](gt)
        if abs(extracted - dv) <= tol:
            return True
    return False


def compute_faithfulness(extraction: dict, gt_contributions: list) -> dict:
    """Compute RA, SA, VA following Eq. 1 from Ichmoukhamedov et al. (2024).
    phi = null/not extracted is removed from the denominator."""
    gt_rank  = {c['feature']: i              for i, c in enumerate(gt_contributions)}
    gt_sign  = {c['feature']: (1 if c['contribution'] >= 0 else -1)
                for c in gt_contributions}
    gt_value = {c['feature']: c['value']     for c in gt_contributions}

    ra_hits, ra_n = 0, 0
    sa_hits, sa_n = 0, 0
    va_hits, va_n = 0, 0

    for feat, info in extraction.items():
        feat_key = feat.lower()
        if feat_key not in gt_rank:
            continue  # feature not in the top K, skip (as in the paper)

        r = info.get('rank')
        if r is not None:
            ra_n += 1
            try:
                if int(float(r)) == gt_rank[feat_key]:
                    ra_hits += 1
            except (ValueError, TypeError):
                pass

        s = info.get('sign')
        if s is not None:
            sa_n += 1
            try:
                if int(float(s)) == gt_sign[feat_key]:
                    sa_hits += 1
            except (ValueError, TypeError):
                pass

        v = info.get('value')
        if v is not None and str(v).lower() not in ('null', 'none', ''):
            try:
                v_float = float(v)
                gt_v    = float(gt_value.get(feat_key, 0))
                va_n += 1
                if is_value_match(feat_key, v_float, gt_v):
                    va_hits += 1
            except (ValueError, TypeError):
                pass

    return {
        'RA': round(ra_hits / ra_n, 4) if ra_n > 0 else None,
        'SA': round(sa_hits / sa_n, 4) if sa_n > 0 else None,
        'VA': round(va_hits / va_n, 4) if va_n > 0 else None,
        'RA_hits': ra_hits, 'RA_n': ra_n,
        'SA_hits': sa_hits, 'SA_n': sa_n,
        'VA_hits': va_hits, 'VA_n': va_n,
        'n_extracted': len(extraction),
    }


faith_rows = []
for _, row in df.iterrows():
    key = f"{row['pipeline']}_{row['xai_model'].lower()}_inst{row['instance_id']}"
    ext = extractions.get(key, {}).get('extraction', {})
    m   = compute_faithfulness(ext, row['gt_contributions'])
    faith_rows.append({
        'pipeline_label': row['pipeline_label'],
        'xai_model':      row['xai_model'],
        'instance_id':    row['instance_id'],
        **m,
    })

faith_df = pd.DataFrame(faith_rows)

summary = faith_df.groupby('pipeline_label')[['RA', 'SA', 'VA']].mean().round(3)
print('Mean faithfulness metrics (Eq. 1, Ichmoukhamedov et al. 2024):')
display(summary)

print()
print('Broken down by model:')
display(faith_df.groupby(['pipeline_label', 'xai_model'])[['RA', 'SA', 'VA']].mean().round(3))

Mean faithfulness metrics (Eq. 1, Ichmoukhamedov et al. 2024):


,RA,SA,VA
pipeline_label,,,
JSON to Text,1.000,1.0,1.0
Tool Use,0.988,1.0,1.0
Vision,0.846,1.0,1.0



Broken down by model:


RA   SA   VA
pipeline_label xai_model                 
JSON to Text   EBM        1.000  1.0  1.0
               XGB        1.000  1.0  1.0
Tool Use       EBM        0.975  1.0  1.0
               XGB        1.000  1.0  1.0
Vision         EBM        0.717  1.0  1.0
               XGB        0.975  1.0  1.0

In [6]:
# Detail table: absolute hit rates
detail = faith_df.groupby('pipeline_label').agg(
    RA_hits=('RA_hits', 'sum'), RA_n=('RA_n', 'sum'),
    SA_hits=('SA_hits', 'sum'), SA_n=('SA_n', 'sum'),
    VA_hits=('VA_hits', 'sum'), VA_n=('VA_n', 'sum'),
    n_extracted=('n_extracted', 'mean'),
)
detail['RA (abs)'] = detail.apply(lambda r: f"{int(r.RA_hits)}/{int(r.RA_n)}", axis=1)
detail['SA (abs)'] = detail.apply(lambda r: f"{int(r.SA_hits)}/{int(r.SA_n)}", axis=1)
detail['VA (abs)'] = detail.apply(lambda r: f"{int(r.VA_hits)}/{int(r.VA_n)}", axis=1)
print('Absolute hits (sum over all 10 narratives per pipeline):')
display(detail[['RA (abs)', 'SA (abs)', 'VA (abs)', 'n_extracted']].rename(
    columns={'n_extracted': 'avg extracted features'}
))

Absolute hits (sum over all 10 narratives per pipeline):


,RA (abs),SA (abs),VA (abs),avg extracted features
pipeline_label,,,,
JSON to Text,80/80,80/80,80/80,5.75
Tool Use,78/79,79/79,78/78,6.05
Vision,66/77,77/77,76/76,5.55


In [7]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
metric_labels = [
    'RA Rank accuracy',
    'SA Sign accuracy',
    'VA Value accuracy',
]
metric_cols = ['RA', 'SA', 'VA']
labels  = [PIPELINE_LABELS[p] for p in PIPELINES]
colors  = ['#4c72b0', '#dd8452', '#55a868']

for ax, col, title in zip(axes, metric_cols, metric_labels):
    vals = [
        faith_df[faith_df.pipeline_label == PIPELINE_LABELS[p]][col].mean()
        for p in PIPELINES
    ]
    bars = ax.bar(labels, vals, color=colors, width=0.5)
    ax.set_ylim(0, 1.1)
    ax.set_title(title, fontsize=10, pad=8)
    ax.set_ylabel('Accuracy (0 to 1)')
    ax.axhline(1.0, color='gray', linestyle='--', alpha=0.35, linewidth=1)
    for bar, val in zip(bars, vals):
        if val is not None:
            ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle(
    'Faithfulness metrics following Ichmoukhamedov et al. (2024), Eq. 1',
    y=1.03, fontsize=12
)
plt.tight_layout()
out_path = RESULTS_DIR / 'eval06_faithfulness.png'
plt.savefig(out_path, dpi=130, bbox_inches='tight')
display(fig)
print(f'Saved: {out_path}')

<Figure size 1300x450 with 3 Axes>

Saved: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/eval06_faithfulness.png


### 4.1 Methodological limitation: selection bias of the metrics

Important note on interpreting RA, SA, VA:

The formula iterates over extraction.items(), so only over features that the LLM
mentioned itself in the narrative. Top K features that the LLM did not mention are
not counted as errors (they are simply absent from the denominator).

Consequence:
* An LLM that mentions only 1 feature and gets its sign right scores SA = 1.0, just
  like an LLM that describes all 4 features correctly.
* The metrics measure precision, not recall of the feature description.
* SA = 1.0 across all pipelines means: what was mentioned had the right sign, not:
  all important features were mentioned.

A fairer approach (not implemented here, to stay paper conform): iterate over
gt_contributions (all top K) with a penalty for unmentioned features. This would in
particular give less of an advantage to JSON to Text (direct access to numbers).

## 5. Assumptions (plausibility)

The extraction LLM extracts one assumption per feature: a sentence that describes the
injected background knowledge of the generation LLM.

Example from the paper (Fig. 3):
> "Scoring goals is a key factor in determining a team's performance and the likelihood of a player standing out."

The paper scores assumptions with perplexity (Eq. 2) relative to Llama 3 8B or Mistral 7B.
Since no local LLM is available, the assumptions are extracted and shown here (perplexity: cell 15).

In [8]:
assumption_rows = []
for _, row in df.iterrows():
    key = f"{row['pipeline']}_{row['xai_model'].lower()}_inst{row['instance_id']}"
    ext = extractions.get(key, {}).get('extraction', {})
    for feat, info in ext.items():
        assumption = info.get('assumption', 'None')
        if assumption and assumption.lower() not in ('none', 'null', ''):
            assumption_rows.append({
                'pipeline_label': row['pipeline_label'],
                'xai_model':      row['xai_model'],
                'instance_id':    row['instance_id'],
                'feature':        feat.lower(),
                'assumption':     assumption,
            })

assumption_df = pd.DataFrame(assumption_rows)
assumption_df.to_csv(OUT_DIR / 'assumptions.csv', index=False)

print(f'Total extracted assumptions: {len(assumption_df)}')
print(f'Mean assumptions per narrative: {len(assumption_df)/len(df):.2f}')
print(f'Saved: {OUT_DIR / "assumptions.csv"}')
print()

# Assumptions per pipeline
print('Mean assumptions per narrative by pipeline:')
display(
    assumption_df.groupby('pipeline_label').size().div(10).rename('avg assumptions').round(1)
)

print()
print('Example assumptions (first 9 entries):')
print('=' * 70)
for _, r in assumption_df.head(9).iterrows():
    print(f"[{r['pipeline_label']:10s} | {r['xai_model']} | inst={r['instance_id']:4d} | {r['feature']}]")
    print(f"  {r['assumption']}")
    print()

Total extracted assumptions: 340
Mean assumptions per narrative: 5.67
Saved: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/eval06_ichmoukhamedov/assumptions.csv

Mean assumptions per narrative by pipeline:


pipeline_label
JSON to Text    11.5
Tool Use        11.4
Vision          11.1
Name: avg assumptions, dtype: float64


Example assumptions (first 9 entries):
[JSON to Text | XGB | inst= 224 | temp]
  Cold temperatures deter people from renting bikes, reducing demand.

[JSON to Text | XGB | inst= 224 | hr]
  Evening commuter peak hours see higher bike rental demand as people travel home from work.

[JSON to Text | XGB | inst= 224 | mnth]
  February is a winter month with typically lower outdoor activity and bike rental demand.

[JSON to Text | XGB | inst= 224 | yr]
  2011 was an earlier, lower-demand phase of the bike-sharing system's history with a smaller user base.

[JSON to Text | XGB | inst= 224 | weekday]
  Thursdays are regular workdays generating commuter traffic that slightly boosts bike rentals.

[JSON to Text | XGB | inst= 224 | hum]
  Moderate humidity is neither too dry nor too damp, having a marginal positive effect on cycling comfort.

[JSON to Text | XGB | inst= 580 | hr]
  Late-night hours see reduced bike rental demand as the evening peak has passed.

[JSON to Text | XGB | inst= 580 |

## 6. Overview

In [9]:
print('=== Evaluation following Ichmoukhamedov et al. (2024): summary ===')
print()

faith_summary = faith_df.groupby('pipeline_label')[['RA', 'SA', 'VA']].mean().round(3)
faith_std     = faith_df.groupby('pipeline_label')[['RA', 'SA', 'VA']].std().round(3)

print('Faithfulness (mean +/- std):')
for pl in [PIPELINE_LABELS[p] for p in PIPELINES]:
    if pl not in faith_summary.index:
        continue
    m = faith_summary.loc[pl]
    s = faith_std.loc[pl]
    def _fmt(col):
        mv = m[col]
        sv = s[col]
        if pd.isna(mv):
            return f'{col}=n/a'
        std_str = f'+/-{sv:.3f}' if not pd.isna(sv) else ''
        return f'{col}={mv:.3f}{std_str}'
    print(f"  {pl:10s}  {_fmt('RA')}  {_fmt('SA')}  {_fmt('VA')}")

print()
print(f'Assumptions extracted: {len(assumption_df)} '
      f'(mean {len(assumption_df)/len(df):.1f} per narrative)')

print()
print('Implementation status:')
print('  Faithfulness (RA, SA, VA)    implemented following Eq. 1')
print('  Assumption extraction        by the extraction LLM (Claude)')

# Save results
faith_df.to_csv(OUT_DIR / 'faithfulness_metrics.csv', index=False)
faith_summary.to_csv(OUT_DIR / 'faithfulness_summary.csv')
print(f'\nFiles saved in: {OUT_DIR}')

=== Evaluation following Ichmoukhamedov et al. (2024): summary ===

Faithfulness (mean +/- std):
  JSON to Text  RA=1.000+/-0.000  SA=1.000+/-0.000  VA=1.000+/-0.000
  Vision      RA=0.846+/-0.251  SA=1.000+/-0.000  VA=1.000+/-0.000
  Tool Use    RA=0.988+/-0.056  SA=1.000+/-0.000  VA=1.000+/-0.000

Assumptions extracted: 340 (mean 5.7 per narrative)

Implementation status:
  Faithfulness (RA, SA, VA)    implemented following Eq. 1
  Assumption extraction        by the extraction LLM (Claude)

Files saved in: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/eval06_ichmoukhamedov
